# Film Boiling Testcase
This is a testcase for evaporation.  
A thin vapor film is situated above a heated wall.  
The flat liquid-vapor interface is disturbed according to the largest rayleigh-taylor instability wavelength.  
As the liquid evaporated a vapor bubble is forming and eventually emerging from the film.  
references for this testcase (in some points they possess some discrepancies)  
`10.1016/j.jcp.2006.07.035` (G)  
`10.1006/jcpn.200.6481` (WW)     

This worksheet also documents the results published in Rieckmann et al. (2024) `10.1016/j.jcp.2023.112716`

Note the following:
* We take the case with $\Delta T = 5K$
* We use the value for $h_v$ given in (WW)

In [ ]:
#r "BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSFE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using Microsoft.AspNetCore.Html;
Init();

In [ ]:
string proj = "Filmboiling_v2";
BoSSSshell.WorkflowMgm.Init(proj);
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

In [ ]:
var sessions = BoSSSshell.WorkflowMgm.DefaultDatabase.Sessions;
sessions

## Grid and Boundary/Initial Value configuration

In [ ]:
static class Constants{
    // careful order of declaration matters!!!
    // material parameters
    public static double rho_A = 200.0;
    public static double rho_B = 5.0;

    public static double mu_A = 0.1;
    public static double mu_B = 0.005;

    public static double Sigma = 0.1;

    public static double c_A = 400.0;
    public static double c_B = 200.0;

    public static double k_A = 40.0;
    public static double k_B = 1.0;

    public static double hVap  = 10000;
    public static double T_sat = 0; // 500K, shifted
    public static double T_wall = T_sat + 5.0;

    public static double alpha_A = k_A / (c_A*rho_A);
    public static double alpha_B = k_B / (c_B*rho_B);
    public static double eps = 1.0-rho_B/rho_A;

    public static double g = 9.81;

    public static double lambda = 2 * Math.PI * Math.Sqrt((3*Sigma)/(g*Math.Abs(rho_A-rho_B)));
    public static double L = lambda / 2.0;

    // capillary timestep , for finest res + highest degree, use this, for comparability?!, Is very small 1e-7 => 1e5 - 1e6 timesteps necessary => artificial surface tension?!
    public static Func<int, int, double> dt = (res, p) => 0.5 * Math.Sqrt((rho_A + rho_B) * Math.Pow(L / ((double)res * ((double)p + 1)), 3) / (2 * Math.PI * Math.Abs(Sigma)));
}

In [ ]:
Constants.L

In (WW) $h_v = 10^4$ in (G) $h_v = 10^3$ which is right?  
Form the plots in (G) we approximate  
$\Delta V(t = 0.455) \approx \Pi * 0.02^2 \approx 1.257*10^{-3}$  
Further:  
$\Delta V(t = 0.455) \approx \frac{\Delta m}{rho_v}$  
$\Delta m(t) \approx \frac{k_v \frac{\Delta T}{\delta}}{h_v}l_{\mathfrak{J}} t \approx 6.283*10^{-3}$
$h_v \approx \frac{k_v \frac{\Delta T}{\delta}}{\Delta m}l_{\mathfrak{J}} 0.455 \approx 11500$  
where:  
$l_{\mathfrak{J}} \approx \lambda$  
$\delta \approx 4/128 \lambda$  
We conclude, that $h_v = 10^4$ has to be used!  

In [ ]:
static class GridFactory{
    public static GridCommons Grid(int res, IDatabaseInfo myDb){
        double h = Constants.L/res;
        double[] Xnodes = GenericBlas.Linspace(-Constants.L, Constants.L, 2 *res + 1);
        double[] Ynodes = GenericBlas.Linspace(0, 6 * Constants.L, 6*res + 1);

        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes);

        grd.EdgeTagNames.Add(1, "freeslip_ZeroGradient");
        grd.EdgeTagNames.Add(2, "pressure_outlet_ConstantTemperature");
        grd.EdgeTagNames.Add(3, "wall_ConstantTemperature");
        
        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 1;
            if((Math.Abs(X[0] - Xnodes.First()) < 1e-6) || (Math.Abs(X[0]- Xnodes.Last()) < 1e-6))
                return 1;
            else if((Math.Abs(X[1] - Ynodes.Last()) < 1e-6) )
                return 2;
            else if((Math.Abs(X[1] - Ynodes.First()) < 1e-6) )
                return 3;
            return et;
        });

        myDb.Controller.DBDriver.SaveGrid(grd, myDb);

        return grd;
    }

    public static GridCommons GridConvergence(int res, IDatabaseInfo myDb){
        double h = Constants.L/res;
        double[] Xnodes = GenericBlas.Linspace(-Constants.L, Constants.L, 2 *res + 1);
        double[] Ynodes = GenericBlas.Linspace(0, 2 * Constants.L, 2 * res + 1);

        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes);

        grd.EdgeTagNames.Add(1, "freeslip_ZeroGradient");
        grd.EdgeTagNames.Add(2, "pressure_outlet_ConstantTemperature");
        grd.EdgeTagNames.Add(3, "wall_ConstantTemperature");
        
        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 1;
            if((Math.Abs(X[0] - Xnodes.First()) < 1e-6) || (Math.Abs(X[0]- Xnodes.Last()) < 1e-6))
                return 1;
            else if((Math.Abs(X[1] - Ynodes.Last()) < 1e-6) )
                return 2;
            else if((Math.Abs(X[1] - Ynodes.First()) < 1e-6) )
                return 3;
            return et;
        });

        myDb.Controller.DBDriver.SaveGrid(grd, myDb);

        return grd;
    }

    public static GridCommons GridConvergencePeriodic(int res, IDatabaseInfo myDb){
        double h = Constants.L/res;
        double[] Xnodes = GenericBlas.Linspace(-Constants.L, Constants.L, 2 *res + 1);
        double[] Ynodes = GenericBlas.Linspace(0, 2 * Constants.L, 2 * res + 1);

        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes, periodicX: true);

        grd.EdgeTagNames.Add(1, "pressure_outlet_ConstantTemperature");
        grd.EdgeTagNames.Add(2, "wall_ConstantTemperature");
        
        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 1;
            if((Math.Abs(X[1] - Ynodes.Last()) < 1e-6) )
                return 1;
            else if((Math.Abs(X[1] - Ynodes.First()) < 1e-6) )
                return 2;
            return et;
        });

        myDb.Controller.DBDriver.SaveGrid(grd, myDb);

        return grd;
    }
}

In [ ]:
public static class BoundaryAndInitialValueFactory { 

    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
            stw.WriteLine("static class BoundaryAndInitialValues {");
            stw.WriteLine("     ");
            stw.WriteLine("     public static Func<double[], double> InterfacePos(){");
            stw.WriteLine("         return X => " + Constants.lambda + " / 128 * (4 + Math.Cos(2 * Math.PI * X[0] / " + Constants.lambda + " ));"); //(G)
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double Phi(double[] X, double t){");
            stw.WriteLine("         return InterfacePos()(X) - X[1];");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double TemperatureB(double[] X, double t){");
            stw.WriteLine("         return " + Constants.T_sat + " + (" + Constants.T_wall + " - " + Constants.T_sat + ") * (1 - X[1]/InterfacePos()(X));");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("}");
            return stw.ToString();
        }
    }
    
    static public Formula Get_TempB() {
        return new Formula("BoundaryAndInitialValues.TemperatureB", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_Phi() {
        return new Formula("BoundaryAndInitialValues.Phi", true, AdditionalPrefixCode:GetPrefixCode());
    }
}

## Initialize Controlfiles

First Simulations on a smaller domain, to see if it convegeces towards some particular shape.

In [ ]:
List<XNSFE_Control> Controls = new List<XNSFE_Control>();

In [ ]:
int res = 2;
int[] amr = { 2, 3, 4};
int[] pDeg = {2, 3};
foreach(int p in pDeg){
    foreach(int lvl in amr){

        XNSFE_Control C = new XNSFE_Control();

        // basic database options
        // ======================
        C.DbPath      = Path.GetFullPath(BoSSSshell.WorkflowMgm.DefaultDatabase.Path);
        C.SessionName = proj + "_H" + res + "_AMR" + lvl + "_P" + p + "_Convergence";
        C.ProjectName = proj;
        C.ProjectDescription = "film boiling - testcase for evaporation";
        
        C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("Degree", p));
        C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("AMR", lvl));
        C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("Res", res));
        C.Paramstudy_CaseIdentification.Add(new Tuple<string, object>("Convtype", "H"));


        // DG degrees
        // ==========
        C.SetDGdegree(p);

        // Physical Parameters
        // ===================
        C.PhysicalParameters.Material = false;

        C.PhysicalParameters.IncludeConvection = true;
        C.ThermalParameters.IncludeConvection  = true;

        C.PhysicalParameters.rho_A = Constants.rho_A;
        C.PhysicalParameters.rho_B = Constants.rho_B;

        C.PhysicalParameters.mu_A = Constants.mu_A;
        C.PhysicalParameters.mu_B = Constants.mu_B;

        C.PhysicalParameters.Sigma = Constants.Sigma;

        C.ThermalParameters.rho_A = Constants.rho_A;
        C.ThermalParameters.rho_B = Constants.rho_B;

        C.ThermalParameters.c_A = Constants.c_A;
        C.ThermalParameters.c_B = Constants.c_B;

        C.ThermalParameters.k_A = Constants.k_A;
        C.ThermalParameters.k_B = Constants.k_B;

        C.ThermalParameters.hVap  = Constants.hVap;
        C.ThermalParameters.T_sat = Constants.T_sat;

        C.PhysicalParameters.betaS_A = 0.0;
        C.PhysicalParameters.betaS_B = 0.0;

        C.PhysicalParameters.betaL      = 0.0;
        C.PhysicalParameters.sliplength = 0.0;
        C.PhysicalParameters.theta_e    = Math.PI / 2.0; 

        // grid generation
        // ===============
        bool periodic = true;
        var grd = periodic ? GridFactory.GridConvergencePeriodic(res, BoSSSshell.WorkflowMgm.DefaultDatabase) : GridFactory.GridConvergence(res, BoSSSshell.WorkflowMgm.DefaultDatabase);
        C.SetGrid(grd);
        
        // Initial Values
        // ==============
        C.AddInitialValue("Phi", BoundaryAndInitialValueFactory.Get_Phi());
        C.AddInitialValue("Temperature#A", "X => "+Constants.T_sat, false);
        C.AddInitialValue("Temperature#B", BoundaryAndInitialValueFactory.Get_TempB());
        C.AddInitialValue("GravityY#A", new Formula($"X => -{Constants.g}",false));
        C.AddInitialValue("GravityY#B", new Formula($"X => -{Constants.g}",false));

        // boundary conditions
        // ===================
        if(!periodic) C.AddBoundaryValue("freeslip_ZeroGradient");
        C.AddBoundaryValue("pressure_outlet_ConstantTemperature", "Temperature#A", "X => "+Constants.T_sat, false);
        C.AddBoundaryValue("wall_ConstantTemperature", "Temperature#B", "X => "+Constants.T_wall, false);


        // level set options
        // =================
        C.Option_LevelSetEvolution                        = BoSSS.Solution.LevelSetTools.LevelSetEvolution.StokesExtension;
        C.Timestepper_LevelSetHandling                    = BoSSS.Solution.XdgTimestepping.LevelSetHandling.LieSplitting;
        C.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.LevelSetTools.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
        C.ReInitPeriod = 20;

        // Timestepping
        // ============
        C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
        C.dtFixed          = Constants.dt((int)(res * Math.Pow(2, amr.Max())), pDeg.Max()); // Use same timestep for all, better comparability
        C.Endtime          = 0.5;
        C.NoOfTimesteps    = (int)Math.Ceiling(C.Endtime /C.dtFixed);
        C.saveperiod       = 10;
        C.rollingSaves     = false;
        
        // AMR
        // ============
        int level = lvl;
        C.AdaptiveMeshRefinement = lvl > 0;
        C.activeAMRlevelIndicators.Add(new BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater.AMRonNarrowband(){ maxRefinementLevel = level});
        C.AMR_startUpSweeps = lvl;

        // special options
        // ===============
        C.NonLinearSolver.MaxSolverIterations = 20;
        C.PostprocessingModules.Add(new BoSSS.Application.XNSFE_Solver.PhysicalBasedTestcases.MassfluxLogging() { LogPeriod = 1 }); // further logging?

        Controls.Add(C);
    }
}

In [ ]:
int ctrlCount = Controls.Count();
Controls.ForEach(c => c.SessionName.Display());
ctrlCount

In [ ]:
var jobs = Controls.Select(c => c.CreateJob()).ToArray();
jobs.ForEach(j => j.NumberOfThreads = 1);
jobs.ForEach(j => j.EnvironmentVars["BOSSS_ARG_7"] = j.NumberOfThreads.ToString());
jobs.Activate();

In [ ]:
BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(172800);

In [ ]:
int count = BoSSSshell.wmg.Sessions.Count();
int success = BoSSSshell.wmg.Sessions.Where(s => s.SuccessfulTermination).Count();

if(count != ctrlCount || count != success){
    // forgive 1 fail
    if(count-1 > success){
        throw new ApplicationException("Not all simulations calculated or finished successful!");
    }
}